# 100→150 유니버스 구조 추천

## tl;dr

- 기술 8개·비기술 42개를 추가해 기술주 종목 수 비중을 35.0%에서 28.7%로 낮춘다.
- 경제적 본거지 기준 미국 30개·비미국 20개를 추가해 최종 미국/비미국 비중을 69.3%/30.7%로 맞춘다.
- 신규 50개는 기존 로더가 지원하는 USD·EUR·JPY·GBP·CHF 상장만 사용한다.
- 실제 신규 데이터는 미적재 상태이므로 이 결과는 구조 추천이며 D1 데이터 게이트가 남아 있다.


## Context & Methods

### Key Assumptions

- 현재 100종목의 섹터와 상태는 `ai_signal_data.xlsx`의 `Universe_Meta`를 그대로 사용한다.
- 국가는 경제적 본거지 기준으로 수동 매핑한다.
- 이전 65→100 확장의 가중치(분산 30%, 데이터 20%, 동종사 20%, 유동성·품질 15%, 독립성 15%)를 유지한다.
- 구조 점수는 기대수익이나 밸류에이션 점수가 아니다.


In [1]:
import json
import sqlite3
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parents[1]
OUT = ROOT / 'outputs' / 'universe_150_recommendation'
RESULTS = json.loads((OUT / 'universe_150_results.json').read_text(encoding='utf-8'))
DB = OUT / 'universe_150_analysis.sqlite'
print('analysis_as_of=', RESULTS['analysis_as_of'])
print('data_approval_status=', RESULTS['data_approval_status'])


analysis_as_of= 2026-07-18
data_approval_status= Share with caveats: structural recommendation ready; new-name workbook coverage not yet loaded


## Data

원본 100종목, 제안 50종목, 섹터·지역·통화 집계와 검증 결과를 SQLite 스냅샷에서 다시 읽는다.


In [2]:
with sqlite3.connect(DB) as connection:
    candidates = pd.read_sql_query('SELECT * FROM candidates ORDER BY selection_order', connection)
    sector_mix = pd.read_sql_query('SELECT * FROM sector_mix ORDER BY final_names DESC, sector', connection)
    region_mix = pd.read_sql_query('SELECT * FROM region_mix ORDER BY final_names DESC, region', connection)
    currency_mix = pd.read_sql_query('SELECT * FROM currency_mix ORDER BY add_names DESC, currency', connection)
print('candidate_rows=', len(candidates))
print('unique_short_tickers=', candidates['simple_ticker'].nunique())
print('markets=', sorted(candidates['market'].unique()))
print('currencies=', sorted(candidates['currency'].unique()))


candidate_rows= 50
unique_short_tickers= 50
markets= ['FP', 'GR', 'JP', 'LN', 'SW', 'US']
currencies= ['CHF', 'EUR', 'GBP', 'JPY', 'USD']


## Results

### 1. 섹터 구성


In [3]:
print(sector_mix.to_string(index=False, formatters={'current_share': lambda x: f'{x:.1%}', 'final_share': lambda x: f'{x:.1%}'}))


                sector  current_names  add_names  final_names current_share final_share
            Technology             35          8           43         35.0%       28.7%
           Industrials             14          7           21         14.0%       14.0%
            Financials             11          6           17         11.0%       11.3%
            Healthcare             10          6           16         10.0%       10.7%
Consumer Discretionary              7          5           12          7.0%        8.0%
      Consumer Staples              5          4            9          5.0%        6.0%
                Energy              5          3            8          5.0%        5.3%
Communication Services              3          3            6          3.0%        4.0%
             Materials              3          3            6          3.0%        4.0%
           Real Estate              4          2            6          4.0%        4.0%
             Utilities          

### 2. 경제적 본거지 구성


In [4]:
print(region_mix.to_string(index=False, formatters={'current_share': lambda x: f'{x:.1%}', 'final_share': lambda x: f'{x:.1%}'}))


             region  current_names  add_names  final_names current_share final_share
      United States             74         30          104         74.0%       69.3%
       Europe ex-UK             13         11           24         13.0%       16.0%
     United Kingdom              7          2            9          7.0%        6.0%
              Japan              2          5            7          2.0%        4.7%
Asia ex-Japan/Korea              2          0            2          2.0%        1.3%
             Canada              0          2            2          0.0%        1.3%
        South Korea              2          0            2          2.0%        1.3%


### 3. 추천 50종목


In [5]:
cols = ['selection_order','ticker','name','sector','country','peer_anchor','structural_score','priority','data_gate']
print(candidates[cols].to_string(index=False))


 selection_order  ticker                        name                 sector        country         peer_anchor  structural_score priority data_gate
               1 QCOM US                    Qualcomm             Technology  United States           AVGO, AMD                88        C    D1 미적재
               2 NXPI US          NXP Semiconductors             Technology    Netherlands           TSLA, TER                93        C    D1 미적재
               3 SNPS US                    Synopsys             Technology  United States                CDNS                88        C    D1 미적재
               4  NOW US                  ServiceNow             Technology  United States      CRM, SAP, ORCL                87        C    D1 미적재
               5 CRWD US                 CrowdStrike             Technology  United States                PANW                88        C    D1 미적재
               6  IBM US                         IBM             Technology  United States     ORCL, SAP, CSCO  

### 4. 검증 체크


In [6]:
checks = pd.Series(RESULTS['checks'], name='passed')
print(checks.to_string())
assert checks.all()
assert len(candidates) == 50
assert candidates['simple_ticker'].nunique() == 50
assert not candidates['simple_ticker'].eq('SAN').any()


current_universe_is_100                       True
all_current_status_available                  True
candidate_count_is_50                         True
candidate_full_tickers_unique                 True
candidate_short_tickers_unique                True
no_overlap_with_current_universe              True
all_candidate_markets_supported               True
all_candidate_currencies_already_supported    True
final_universe_is_150                         True
candidate_us_is_30                            True
candidate_non_us_is_20                        True
final_us_share_near_prior_posture             True
technology_share_below_30pct                  True
no_san_collision                              True
score_bounds_valid                            True


## Takeaways

1. 50개 추가안은 이전 확장의 70대30 지역 기조를 유지하면서 기술주 종목 수 비중을 30% 아래로 낮춘다.
2. 보험·거래소·생명과학 도구·의료기기·물류·규제형 유틸리티·주거/헬스케어 REIT·금광의 동종사 밀도가 개선된다.
3. 모든 후보가 현행 접미사·통화 범위 안에 있지만 신규 워크북 데이터는 아직 검증되지 않았다.
4. 유니버스 단일 arm으로 사전등록하고 `val_window` 등 다른 축은 고정해야 효과를 해석할 수 있다.
